# Vienna Bike Availability Prediction

This notebook documents our workflow for predicting the number of available bikes at Vienna nextbike stations.

Final model: CatBoost  
Best Kaggle RMSLE: 1.05697

In [2]:
library(data.table)
library(lubridate)
library(catboost)

## 1. Load Data

The original train and test files are stored locally in the `data/` folder. They are not uploaded to GitHub because of file size limits.

In [3]:
train_path <- "../data/train.csv"
test_path  <- "../data/test.csv"

submission_path <- "../submissions/submission_catboost_baseline.csv"
submission_required_path <- "../submission.csv"

rmsle_fun <- function(actual, pred) {
  pred <- pmax(pred, 0)
  sqrt(mean((log1p(pred) - log1p(actual))^2))
}

train <- fread(train_path)
test  <- fread(test_path)

cat("Train rows:", nrow(train), "\n")
cat("Test rows:", nrow(test), "\n")

head(train)

Train rows: 2150250 
Test rows: 537445 


datetime,station_number,name,lat,lng,bikes
<dttm>,<int>,<chr>,<dbl>,<dbl>,<int>
2024-09-03 17:30:00,32000,Julius-Raab-Platz,48.21154,16.38237,25
2024-09-03 17:30:00,32001,Hoher Markt,48.21067,16.37298,14
2024-09-03 17:30:00,32002,Oper,48.20268,16.36970,9
2024-09-03 17:30:00,32003,Volksgarten,48.20652,16.36040,3
2024-09-03 17:30:00,32004,Taborstraße U2,48.21952,16.38222,5
2024-09-03 17:30:00,32005,WU-Campus / Südportalstraße,48.21286,16.40767,10


## 2. Datetime Processing

In [4]:
train[, datetime := as.POSIXct(datetime, format = "%Y-%m-%dT%H:%M:%SZ", tz = "UTC")]
test[, datetime  := as.POSIXct(datetime,  format = "%Y-%m-%dT%H:%M:%SZ", tz = "UTC")]

cat("Train range:", as.character(min(train$datetime)), "to", as.character(max(train$datetime)), "\n")
cat("Test range:", as.character(min(test$datetime)), "to", as.character(max(test$datetime)), "\n")

Train range: 2024-09-03 17:30:00 to 2025-03-13 08:00:00 
Test range: 2025-03-13 08:30:00 to 2025-04-29 23:30:00 


## 3. Basic Data Exploration

In [5]:
cat("Number of stations in train:", uniqueN(train$station_number), "\n")
cat("Number of stations in test:", uniqueN(test$station_number), "\n")

summary(train$bikes)

Number of stations in train: 235 
Number of stations in test: 235 


   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
   0.00    5.00   11.00   12.35   18.00   78.00 

## 4. Feature Engineering

We create time-based features and historical station-level demand features.

In [6]:
train[, is_train := 1]
test[, is_train := 0]
test[, bikes := NA_integer_]

all_data <- rbindlist(list(train, test), fill = TRUE)

all_data[, `:=`(
  hour = hour(datetime),
  minute = minute(datetime),
  weekday = wday(datetime),
  day = day(datetime),
  month = month(datetime),
  week = isoweek(datetime)
)]

all_data[, `:=`(
  is_weekend = as.integer(weekday %in% c(1, 7)),
  hour_sin = sin(2 * pi * hour / 24),
  hour_cos = cos(2 * pi * hour / 24),
  weekday_sin = sin(2 * pi * weekday / 7),
  weekday_cos = cos(2 * pi * weekday / 7)
)]

head(all_data)

datetime,station_number,name,lat,lng,bikes,is_train,hour,minute,weekday,day,month,week,is_weekend,hour_sin,hour_cos,weekday_sin,weekday_cos
<dttm>,<int>,<chr>,<dbl>,<dbl>,<int>,<dbl>,<int>,<int>,<dbl>,<int>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
2024-09-03 17:30:00,32000,Julius-Raab-Platz,48.21154,16.38237,25,1,17,30,3,3,9,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689
2024-09-03 17:30:00,32001,Hoher Markt,48.21067,16.37298,14,1,17,30,3,3,9,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689
2024-09-03 17:30:00,32002,Oper,48.20268,16.36970,9,1,17,30,3,3,9,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689
2024-09-03 17:30:00,32003,Volksgarten,48.20652,16.36040,3,1,17,30,3,3,9,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689
2024-09-03 17:30:00,32004,Taborstraße U2,48.21952,16.38222,5,1,17,30,3,3,9,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689
2024-09-03 17:30:00,32005,WU-Campus / Südportalstraße,48.21286,16.40767,10,1,17,30,3,3,9,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689


## 5. Historical Station Features

The model uses station-level historical summaries and the average bike count for each station-weekday-hour combination.

In [7]:
train_base <- all_data[is_train == 1]

global_mean <- mean(train_base$bikes, na.rm = TRUE)

station_stats <- train_base[, .(
  station_mean_bikes = mean(bikes, na.rm = TRUE),
  station_max_bikes = max(bikes, na.rm = TRUE),
  station_sd_bikes = sd(bikes, na.rm = TRUE)
), by = station_number]

station_weekday_hour <- train_base[, .(
  avg_bikes_station_weekday_hour = mean(bikes, na.rm = TRUE)
), by = .(station_number, weekday, hour)]

all_data <- merge(
  all_data,
  station_stats,
  by = "station_number",
  all.x = TRUE,
  sort = FALSE
)

all_data <- merge(
  all_data,
  station_weekday_hour,
  by = c("station_number", "weekday", "hour"),
  all.x = TRUE,
  sort = FALSE
)

all_data[is.na(station_mean_bikes), station_mean_bikes := global_mean]
all_data[is.na(station_max_bikes), station_max_bikes := global_mean]
all_data[is.na(station_sd_bikes), station_sd_bikes := 0]
all_data[is.na(avg_bikes_station_weekday_hour), avg_bikes_station_weekday_hour := global_mean]

head(all_data)

station_number,weekday,hour,datetime,name,lat,lng,bikes,is_train,minute,⋯,week,is_weekend,hour_sin,hour_cos,weekday_sin,weekday_cos,station_mean_bikes,station_max_bikes,station_sd_bikes,avg_bikes_station_weekday_hour
<int>,<dbl>,<int>,<dttm>,<chr>,<dbl>,<dbl>,<int>,<dbl>,<int>,⋯,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
32000,3,17,2024-09-03 17:30:00,Julius-Raab-Platz,48.21154,16.38237,25,1,30,⋯,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689,9.258033,38,5.550312,10.981818
32001,3,17,2024-09-03 17:30:00,Hoher Markt,48.21067,16.37298,14,1,30,⋯,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689,16.898142,40,7.105678,18.927273
32002,3,17,2024-09-03 17:30:00,Oper,48.20268,16.36970,9,1,30,⋯,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689,14.598361,51,8.117019,17.127273
32003,3,17,2024-09-03 17:30:00,Volksgarten,48.20652,16.36040,3,1,30,⋯,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689,7.415082,31,5.828585,9.527273
32004,3,17,2024-09-03 17:30:00,Taborstraße U2,48.21952,16.38222,5,1,30,⋯,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689,12.433880,38,5.925758,10.563636
32005,3,17,2024-09-03 17:30:00,WU-Campus / Südportalstraße,48.21286,16.40767,10,1,30,⋯,36,0,-0.9659258,-0.258819,0.4338837,-0.9009689,13.590383,55,9.208178,16.181818


## 6. Prepare Features for CatBoost

CatBoost can directly use categorical variables. We convert station and time category columns into factors.

In [8]:
features <- c(
  "station_number",
  "lat",
  "lng",
  "hour",
  "minute",
  "weekday",
  "day",
  "month",
  "week",
  "is_weekend",
  "hour_sin",
  "hour_cos",
  "weekday_sin",
  "weekday_cos",
  "station_mean_bikes",
  "station_max_bikes",
  "station_sd_bikes",
  "avg_bikes_station_weekday_hour"
)

model_data <- as.data.table(all_data[, ..features])

cat_features <- c(
  "station_number",
  "hour",
  "weekday",
  "month",
  "week",
  "is_weekend"
)

for (col in cat_features) {
  model_data[[col]] <- as.factor(model_data[[col]])
}

for (col in names(model_data)) {
  if (!(col %in% cat_features)) {
    med_val <- median(model_data[[col]], na.rm = TRUE)
    if (is.na(med_val)) med_val <- 0
    model_data[[col]][is.na(model_data[[col]])] <- med_val
  }
}

train_rows <- which(all_data$is_train == 1)
test_rows  <- which(all_data$is_train == 0)

train_data <- as.data.frame(model_data[train_rows])
test_data  <- as.data.frame(model_data[test_rows])

train_final <- all_data[train_rows]
test_final  <- all_data[test_rows]

y_train <- log1p(train_final$bikes)

cat_features_idx <- which(names(train_data) %in% cat_features)

cat("CatBoost categorical feature indices:\n")
print(cat_features_idx)

cat("CatBoost categorical feature names:\n")
print(names(train_data)[cat_features_idx])

cat("Train data:", nrow(train_data), "x", ncol(train_data), "\n")
cat("Test data:", nrow(test_data), "x", ncol(test_data), "\n")

CatBoost categorical feature indices:
[1]  1  4  6  8  9 10
CatBoost categorical feature names:
[1] "station_number" "hour"           "weekday"        "month"         
[5] "week"           "is_weekend"    
Train data: 2150250 x 18 
Test data: 537445 x 18 


## 7. Validation Split

A time-based split is used because the task is a forecasting problem.

In [9]:
cutoff_date <- as.POSIXct("2025-02-20 00:00:00", tz = "UTC")

tr_idx <- which(train_final$datetime < cutoff_date)
va_idx <- which(train_final$datetime >= cutoff_date)

cat("Validation train rows:", length(tr_idx), "\n")
cat("Validation rows:", length(va_idx), "\n")

Validation train rows: 1909375 
Validation rows: 240875 


## 8. Train CatBoost Validation Model

The target is transformed with `log1p`. Optimizing RMSE on the transformed target corresponds closely to RMSLE on the original scale.

In [10]:
train_pool <- catboost.load_pool(
  data = train_data[tr_idx, ],
  label = y_train[tr_idx],
  cat_features = cat_features_idx
)

valid_pool <- catboost.load_pool(
  data = train_data[va_idx, ],
  label = y_train[va_idx],
  cat_features = cat_features_idx
)

params <- list(
  loss_function = "RMSE",
  eval_metric = "RMSE",
  iterations = 2500,
  learning_rate = 0.035,
  depth = 6,
  l2_leaf_reg = 6,
  random_strength = 1,
  bagging_temperature = 0.5,
  od_type = "Iter",
  od_wait = 100,
  random_seed = 123,
  verbose = 100
)

set.seed(123)

valid_model <- catboost.train(
  learn_pool = train_pool,
  test_pool = valid_pool,
  params = params
)

Parameter 'cat_features' is meaningless because column types are taken from data.frame.
Please, convert categorical columns to factors manually.
Parameter 'cat_features' is meaningless because column types are taken from data.frame.
Please, convert categorical columns to factors manually.
0:	learn: 0.9187556	test: 0.8641124	best: 0.8641124 (0)	total: 3.22s	remaining: 2h 14m 15s
100:	learn: 0.5611123	test: 0.6622923	best: 0.6617381 (95)	total: 2m 4s	remaining: 49m 27s
200:	learn: 0.4765783	test: 0.6544649	best: 0.6544649 (200)	total: 3m 49s	remaining: 43m 40s
300:	learn: 0.4586099	test: 0.6505602	best: 0.6504418 (295)	total: 5m 33s	remaining: 40m 38s
400:	learn: 0.4477821	test: 0.6460254	best: 0.6459884 (394)	total: 7m 32s	remaining: 39m 28s
500:	learn: 0.4409872	test: 0.6424125	best: 0.6424125 (500)	total: 9m 44s	remaining: 38m 52s
600:	learn: 0.4353191	test: 0.6401088	best: 0.6399329 (533)	total: 11m 55s	remaining: 37m 41s
700:	learn: 0.4303939	test: 0.6407159	best: 0.6397961 (612)	to

## 9. Validation Performance

In [11]:
valid_pred_log <- catboost.predict(valid_model, valid_pool)
valid_pred <- pmax(expm1(valid_pred_log), 0)

valid_rmsle <- rmsle_fun(train_final$bikes[va_idx], valid_pred)

cat("CatBoost validation RMSLE:", valid_rmsle, "\n")

best_iter <- valid_model$tree_count_

if (is.null(best_iter) || length(best_iter) == 0 || is.na(best_iter)) {
  best_iter <- params$iterations
}

cat("Best/tree count used:", best_iter, "\n")

CatBoost validation RMSLE: 0.6397961 
Best/tree count used: 2500 


## 10. Train Final Model

In [12]:
full_pool <- catboost.load_pool(
  data = train_data,
  label = y_train,
  cat_features = cat_features_idx
)

final_params <- params
final_params$iterations <- as.integer(best_iter)
final_params$od_type <- NULL
final_params$od_wait <- NULL

set.seed(123)

final_model <- catboost.train(
  learn_pool = full_pool,
  params = final_params
)

Parameter 'cat_features' is meaningless because column types are taken from data.frame.
Please, convert categorical columns to factors manually.


: 

## 11. Predict Test Set

In [ ]:
test_pool <- catboost.load_pool(
  data = test_data,
  cat_features = cat_features_idx
)

test_pred_log <- catboost.predict(final_model, test_pool)
test_pred <- pmax(expm1(test_pred_log), 0)

summary(test_pred)
quantile(test_pred, probs = c(0, 0.01, 0.05, 0.5, 0.95, 0.99, 1))

## 12. Create Submission File

The target is a count of bikes, so predictions are rounded to non-negative integers.

In [ ]:
submission <- data.frame(
  id = paste(
    format(test$datetime, "%Y-%m-%d %H:%M:%S", tz = "UTC"),
    test$station_number,
    sep = "_"
  ),
  bikes = as.integer(round(pmax(test_pred, 0)))
)

cat("Submission checks:\n")
cat("Rows:", nrow(submission), "\n")
cat("Expected rows:", nrow(test), "\n")
cat("Duplicate IDs:", sum(duplicated(submission$id)), "\n")
cat("Missing predictions:", sum(is.na(submission$bikes)), "\n")
cat("Negative predictions:", sum(submission$bikes < 0), "\n")

stopifnot(nrow(submission) == nrow(test))
stopifnot(sum(duplicated(submission$id)) == 0)
stopifnot(sum(is.na(submission$bikes)) == 0)
stopifnot(sum(submission$bikes < 0) == 0)

write.csv(submission, submission_path, row.names = FALSE)
write.csv(submission, submission_required_path, row.names = FALSE)

cat("Saved submission files:\n")
cat(submission_path, "\n")
cat(submission_required_path, "\n")

## 13. Conclusion

CatBoost gave the strongest performance in our experiments. Compared with XGBoost, it handled categorical station and time features more effectively.

The notebook generates `submission.csv`, which is the required final prediction file.